<a href="https://colab.research.google.com/github/DivyaS-617/AD-Drug-Insights-RAG/blob/main/AI_Powered_HR_Assistant_Chatbotpynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install llama_index

In [4]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
openai.api_key = userdata.get('openai')

In [5]:
!pip install llama-index-readers-file

from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data_new",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:1000])

9
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This handbook explains workplace expectations, benefits, and policies. If any local law conflicts with
this handbook, applicable law prevails.
2. Company Values & Ways of Working
 Build with clarity: define the user, problem, inputs/outputs, and definition of done.
 Bias for action: ship small, iterate fast, measure outcomes.
 Respect and inclusion: disagreement is allowed; disrespect is not.
 Data

In [6]:
import os
import time
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# Start timer for index setup
start_time = time.time()

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data_new").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

setup_duration = time.time() - start_time
print(f"Index setup time: {setup_duration:.2f} seconds")

# Start timer for query
query_start_time = time.time()

# Prepare the query engine
retriever = VectorIndexRetriever(index=index, similarity_top_k=2)
query_engine = RetrieverQueryEngine(retriever=retriever)

# Execute query
response = query_engine.query("Who all mentioned in the doc?")
print(response)

query_duration = time.time() - query_start_time
print(f"Query time: {query_duration:.2f} seconds")

Index setup time: 5.29 seconds
Employees, managers, HR/People Partner, stakeholders, and qualified counsel.
Query time: 2.24 seconds


In [11]:
import os
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

import time
start_time = time.time()
query_engine = index.as_query_engine()

retriever = VectorIndexRetriever(index=index, similarity_top_k=3)

query_engine = RetrieverQueryEngine(retriever=retriever)

response = query_engine.query("What is DDS maternity policy?")
print(response)


end_time = time.time()  # Record end time
execution_time = end_time - start_time  # Calculate execution time
print(f"Execution time: {execution_time} seconds")

DDS maternity policy follows UAE labor requirements and contract terms. HR provides case-specific guidance confidentially, and managers are required to support a respectful transition plan before and after the leave.
Execution time: 1.5827598571777344 seconds


In [14]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Define a system prompt
system_prompt = '''
You are MEME, the Enterprise HR chatbot for Decoding Data Science (DDS). Respond only to questions related to Human Resources, including company policies, leave, payroll, employee benefits, and other HR topics.

- If the inquiry is outside your HR domain, reply politely: "I can only assist with HR-related queries."
- If the question is unclear or you're unsure of its meaning, ask a clarifying question to obtain more detail.
- If after clarification you still cannot address the question, reply: "Please contact our HR team at connect@decodingdatascience.com."
- Always maintain a polite, professional, and concise tone.
- Do not provide information on non-HR topics under any circumstance.
- If the user asks multiple questions, respond to HR-related parts only, following the same guidance above for the rest.

Output Format:
- Respond in a short, polite paragraph (typically 1-3 sentences), addressing only the HR component or following the prescribed fallback messages when appropriate.

Examples:

Example 1
User: How much annual leave do I get?
Response:
Employees at DDS receive 18 days of paid annual leave per year. If you have further questions about your leave balance or application process, please let me know.

Example 2
User: Can you reset my laptop password?
Response:
I can only assist with HR-related queries.

Example 3
User: What's covered under our health insurance? Also, when is the next company hackathon?
Response:
Our health insurance plan covers medical, dental, and vision care for employees and their immediate families. For information about company events, I can only assist with HR-related queries.

Example 4
User: I need help, but I'm not sure what department to contact.
Response:
Could you please clarify your query so I can determine if I can assist you with an HR-related concern? If it falls outside HR, I will guide you to the correct contact.

Example 5
User: [writes something ambiguous or unclear]
Response:
Could you please provide more details or clarify your question so I can assist you?

Example 6
User: [after clarifying, question is still not HR or is ambiguous]
Response:
Please contact our HR team at connect@decodingdatascience.com.

(For realistic interactions, responses should mirror these examples in length and style, adapting the HR answer or fallback as appropriate.)

---

**Important:**
Your objective is to answer only HR-related questions politely, professionally, and concisely as MEME, the DDS HR chatbot. If the question falls outside your scope or remains unclear, use the specified fallback responses.
'''
documents = SimpleDirectoryReader("data_new").load_data()

index = VectorStoreIndex.from_documents(documents=documents)

# Configure query engine with system prompt
query_engine = index.as_query_engine(system_prompt=system_prompt)

response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)


The standard office hours in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.


In [16]:
import time
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Start timer
start_time = time.time()

# Load and index documents
documents = SimpleDirectoryReader("data_new").load_data()
index = VectorStoreIndex.from_documents(documents=documents)

# Query the index
query_engine = index.as_query_engine()
response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)

# End timer and print duration
end_time = time.time()
print(f"\nExecution Time: {end_time - start_time:.2f} seconds")

The standard office hours in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.

Execution Time: 2.87 seconds


In [12]:
pip install gradio

In [22]:
import gradio as gr
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Load data and build the index
documents = SimpleDirectoryReader("data_new").load_data()
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine()

# Function to handle queries
def query_document(query):
    response = query_engine.query(query)
    return f"Hi, I am MEME. How can I help you?\n\n{str(response)}"

# Gradio interface
interface = gr.Interface(
    fn=query_document,
    inputs=gr.Textbox(label="Enter your query", placeholder="Type your question here..."),
    outputs=gr.Textbox(label="Response"),
    title="MEME - AI-Powered HR Assistant Chatbot",
    description="Ask questions about the documents loaded into the system."
)

# Launch the Gradio app
if __name__ == "__main__":
    interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5793faf9899acc048a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
